In [10]:
library(Seurat)
library(dplyr)

setwd('/Aging/Data/Anderson/Multiomics/')
DIR <- '/Aging/Data/Anderson/Multiomics/ATAC/ArchR/'
data_dir2 <- '/Aging/Data/Anderson/snRNA_Seq/'

# Step 5

# 1. Load tables 
Exc_Ctrl <- read.csv(paste0(DIR,'Anderson_Exc_Ctrl_PeakMatrix_reads_RoadMap_18States.csv'), check.names=FALSE)
Exc_AD <- read.csv(paste0(DIR,'Anderson_Exc_AD_PeakMatrix_reads_RoadMap_18States.csv'), check.names=FALSE)
Exc <- full_join(Exc_Ctrl, Exc_AD, by='Description')
rownames(Exc) <- Exc$Description

Inh <- read.csv(paste0(DIR,'Anderson_Inh_PeakMatrix_reads_RoadMap_18States.csv'), check.names=FALSE)

df <- full_join(Exc, Inh, by='Description')
rownames(df) <- df$Description
df <- df[,-which(names(df) == 'Description')]
df <- t(df) # cells in rows / states in columns

# 2. Center log-ratio Normalization (across cells / per state)
clr_function <- function(x, pseudocount = 1) {
  x <- x + pseudocount               # Avoid log(0)
  log_x <- log(x)                    # Log transform
  clr_x <- log_x - mean(log_x, na.rm=T)       # Centered log-ratio
  return(clr_x)
}
clr_matrix <- apply(df, 2, clr_function) # applied across rows (cells) / column (state) 

# 3. Deviation to the mean of that ChromHMM state (already done / not necessary)
#clr_matrix <- sweep(clr_matrix, 2, colMeans(clr_matrix, na.rm = TRUE), FUN = "-")

# 4. Assign weights -- Active states (Enhancers; TSS; Tx) assigned negative sign, and the repressive regions (Quies, Repr, Het, TxWk) assigned positive sign
# transpose 
clr_matrix <- as.data.frame(t(clr_matrix)) # cells in columns / states in rows
clr_matrix$Weight <- ifelse(
  grepl('Quies|TxWk|Repr|Het', rownames(clr_matrix)), 1, -1)
clr_matrix[1:(length(clr_matrix)-1)] <- clr_matrix[1:(length(clr_matrix)-1)]*clr_matrix$Weight
clr_matrix <- clr_matrix[1:(length(clr_matrix)-1)]

# 5. Summing up Deviations from 18 ChromHMM states / cell 
# Higher score, Repressive regions more open, Higher erosion 
ER_score <- as.data.frame(colSums(clr_matrix, na.rm=T))
colnames(ER_score)='Epigenetic_Erosion_Score'
#ER_score$X <- rownames(ER_score)

write.csv(ER_score, paste0(data_dir2,"GSE214979_ER_score.csv"))